# 長江杯 2026 — A100 Colab 自動跑批與斷點續跑 Worker（Ultra 5x 1,000 CU 省額度版）

**核心防中斷與省額度特性**：
- 🔄 **單序列級別斷點續跑 (`Checkpoint & Resume`)**：每跑完 1 條序列（約 5–10 秒）立即寫入 Google Drive；若中途網路斷線或瀏覽器休眠，重連後直接跑 **第 1 格 + 第 4 格**，已算完的序列會 0 秒跳過 (`[SKIP 斷點續跑]`)，自動從中斷處繼續！
- 📡 **即時串流輸出（防 WebSocket 閒置斷線）**：全程即時印出每條序列進度，避免 Colab 因長時間無輸出而判定閒置斷線。
- 🛡️ **智慧釋放 GPU（絕不誤踢）**：僅當**至少 1 個真實 GPU 任務全部成功跑完**且確認寫入 Google Drive 後，才會倒數 15 秒並執行 `runtime.unassign()`；若隊列為空、中途報錯或你手動按停止，**絕不踢掉 GPU**。

In [ ]:
#@title 1. 檢查 GPU 架構、掛載 Google Drive、同步 GitHub 並檢查斷點續跑狀態（斷線重連後先跑這格）
import os, subprocess, pathlib, torch
from google.colab import drive

DRIVE_WORKSPACE = "/content/drive/MyDrive/Changjiang_Cup_2026" #@param {type:"string"}
GITHUB_REPO_URL = "https://github.com/David-Liv/changjiang-protein-2026.git" #@param {type:"string"}
REPO_DIR = "/content/changjiang-protein-2026"

print("=== [1/4] GPU 架構與 1,000 CU 額度自檢 ===")
!nvidia-smi
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    gb = p.total_memory / 1e9
    sm = f"sm_{p.major}{p.minor}"
    print(f"檢測到 GPU: {p.name} | 顯存: {gb:.1f} GB | 架構: {sm}")
    if p.major >= 12 or "BLACKWELL" in p.name.upper() or "RTX PRO 6000" in p.name.upper():
        print("❌ [警告] 當前為 G4 (Blackwell sm_120)！BindCraft (CUDA 12.4/JAX) 不支持此架構，請切換為 A100！")
    elif "H100" in p.name.upper() or p.major == 9:
        print("⚠️ [提示] 當前為 H100 (~18 CU/h，1,000 CU 約可跑 55 小時)。日常跑批建議改選 A100 以節省額度。")
    else:
        print("✅ [最佳配置] A100 架構完全兼容 ESMFold 與 BindCraft（~5.4–7.5 CU/h，1,000 CU 可跑 135–185 小時）✓")
else:
    print("❌ 未檢測到 GPU！請在『執行階段 -> 變更執行階段類型』選擇 A100 GPU。")

print("\n=== [2/4] 檢查 ESMFold / ipSAE 評估依賴（不升級 numpy，防止內核崩潰）===")
!pip install -q transformers accelerate

print("\n=== [3/4] 掛載 Google Drive ===")
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
pathlib.Path(DRIVE_WORKSPACE).mkdir(parents=True, exist_ok=True)

print("\n=== [4/4] 同步 GitHub 最新 Pipeline 代碼 ===")
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!python3 scripts/tnf_pipeline.py selftest

import sys
sys.path.insert(0, REPO_DIR)
from pipeline.worker import ensure_drive_workspace
dirs = ensure_drive_workspace(pathlib.Path(DRIVE_WORKSPACE))
running = list(dirs['running'].glob('*.json'))
pending = list(dirs['pending'].glob('*.json'))
print(f"\n📂 網盤隊列狀態：待續跑 (running) = {len(running)} 個 | 待執行 (pending) = {len(pending)} 個")
if running:
    print(f"   🔄 檢測到上次未跑完的任務 {[p.name for p in running]}，直接運行『第 4 格』即可從斷點繼續！")

In [ ]:
#@title 2. 安裝 BindCraft + PyRosetta + SolubleMPNN 環境（僅跑靶點 1 / 5 生成任務時需要；若只跑 TNF-α 校準請跳過此格！）
import os, torch
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 12:
    raise RuntimeError("當前 GPU 為 G4 (Blackwell sm_120)，與 BindCraft CUDA 12.4 不兼容！請先切換執行階段為 A100 再安裝。")

# 安裝獨立 Miniforge（不重啟 Colab 內核，避免 condacolab 導致會話閃退）
if not os.path.exists("/content/miniforge3/bin/conda"):
    print("正在安裝 Miniforge3（不重啟內核）...")
    !wget -q -O /content/Miniforge3.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
    !bash /content/Miniforge3.sh -b -p /content/miniforge3
os.environ["PATH"] = "/content/miniforge3/bin:" + os.environ.get("PATH", "")

if not os.path.exists("/content/BindCraft"):
    !git clone https://github.com/martinpacesa/BindCraft /content/BindCraft
    !bash /content/BindCraft/install_bindcraft.sh --cuda '12.4' --pkg_manager 'conda'
else:
    print("BindCraft 已安裝 ✓")

In [ ]:
#@title 3. 將預設任務模板投遞到網盤 `jobs/pending/`（若上次任務已進 `running` 則直接跑第 4 格即可）
import shutil, pathlib

TEMPLATE_CHOICE = "target2_TNFa_calibration" #@param ["target2_TNFa_calibration", "target5_TL1A_minibinder", "target1_EGFR_vhh", "none"]
pending_dir = pathlib.Path(DRIVE_WORKSPACE) / "jobs" / "pending"
running_dir = pathlib.Path(DRIVE_WORKSPACE) / "jobs" / "running"
pending_dir.mkdir(parents=True, exist_ok=True)
running_dir.mkdir(parents=True, exist_ok=True)

mapping = {
    "target2_TNFa_calibration": "/content/changjiang-protein-2026/jobs/templates/job_target2_TNFa_calibration.json",
    "target5_TL1A_minibinder": "/content/changjiang-protein-2026/jobs/templates/job_target5_TL1A_minibinder.json",
    "target1_EGFR_vhh": "/content/changjiang-protein-2026/jobs/templates/job_target1_EGFR_vhh.json",
}
if TEMPLATE_CHOICE in mapping:
    src = pathlib.Path(mapping[TEMPLATE_CHOICE])
    if (running_dir / src.name).exists():
        print(f"🔄 任務 {src.name} 已在 jobs/running/ 中，直接運行『第 4 格』即可從斷點繼續！")
    else:
        dst = pending_dir / src.name
        shutil.copy2(src, dst)
        print(f"✅ 已投遞任務單至隊列：{dst}")

In [ ]:
#@title 4. 啟動隊列執行器（支援斷點續跑 + 即時進度輸出 + 全部成功後倒數 15 秒自動釋放 GPU）
import os

AUTO_UNASSIGN_GPU = True #@param {type:"boolean"}
os.environ["PATH"] = "/content/miniforge3/bin:" + os.environ.get("PATH", "")

!git -C /content/changjiang-protein-2026 pull --ff-only
unassign_flag = "--unassign-on-exit" if AUTO_UNASSIGN_GPU else ""
!python3 -u /content/changjiang-protein-2026/pipeline/worker.py --drive-root "{DRIVE_WORKSPACE}" --once {unassign_flag}